# 2-AMALIYOT: VECTORS EMBEDDINGS (WTE, WPE VS ROPE) & TENZOR SHAPES
**Maqsadi:** Model ichidagi `wte` (Word Token Embedding) va `wpe` (Word Position Embedding) matritsalarini tekshirish hamda **"Odam kitob o'qidi"** va **"Kitob o'qidi odam"** gaplaridagi vektorlar va Cosine Similarity o'zgarishini matematik tahlil qilish.


In [1]:
#  Importlar va Model yuklash
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# GPT-2 model va tokenizerini yuklaymiz
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModel.from_pretrained("gpt2")

print("--- GPT-2 MODEL EMBEDDING QATLAMTUZILISHI ---")
print("Word Token Embedding (wte):", model.wte)
print("Word Position Embedding (wpe):", model.wpe)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

c:\Users\sharg\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sharg\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

--- GPT-2 MODEL EMBEDDING QATLAMTUZILISHI ---
Word Token Embedding (wte): Embedding(50257, 768)
Word Position Embedding (wpe): Embedding(1024, 768)


### 2-BOSQICH: Tenzor Shakllarini (`shape`) Qadambaqadam Tahlil Qilish
Input text $\to$ Token IDs $\to$ Embeddings Matrix $\to$ `torch.Size([batch, seq_len, hidden_dim])`


In [2]:
#  Tenzor shaklini o'rganish
sample_text = "Deep Learning va LLMlar"

# Tokenizatsiya va PyTorch Tensor holatiga o'tkazish
inputs = tokenizer(sample_text, return_tensors="pt")
input_ids = inputs["input_ids"]

# Wte (Token Embedding) matritsasidan o'tkazish
token_embeddings = model.wte(input_ids)

print(f"Kiritilgan matn: '{sample_text}'")
print(f"Token IDs ({input_ids.shape}):", input_ids)
print(f"Vektor Embedding Shakli (Shape): {token_embeddings.shape}")
print("\n--- TENZOR O'LCHAMLARI TAHLILI ---")
print(f"Batch Size (Guruh hajmi): {token_embeddings.shape[0]}")
print(f"Sequence Length (Tokenlar soni): {token_embeddings.shape[1]}")
print(f"Hidden Dimension (d_model): {token_embeddings.shape[2]}")


Kiritilgan matn: 'Deep Learning va LLMlar'
Token IDs (torch.Size([1, 6])): tensor([[29744, 18252, 46935, 27140,    44, 21681]])
Vektor Embedding Shakli (Shape): torch.Size([1, 6, 768])

--- TENZOR O'LCHAMLARI TAHLILI ---
Batch Size (Guruh hajmi): 1
Sequence Length (Tokenlar soni): 6
Hidden Dimension (d_model): 768


### 3-BOSQICH: Wte + Wpe Qo'shilishi va Vektor Elementlarini Ko'rish
To'liq Input Vector hosil bo'lishi:
$$\text{Final Input Vector} = \text{Wte}(\text{Token ID}) + \text{Wpe}(\text{Position Index})$$


In [3]:
#  Wte + Wpe yig'indisi
seq_len = input_ids.shape[1]
position_ids = torch.arange(0, seq_len, dtype=torch.long).unsqueeze(0)

# Wpe (Position Embedding) qiymatlarini olish
position_embeddings = model.wpe(position_ids)

# Ikkala embeddingni qo'shish
final_embeddings = token_embeddings + position_embeddings

print(f"wte (Token Vector) shape: {token_embeddings.shape}")
print(f"wpe (Position Vector) shape: {position_embeddings.shape}")
print(f"Final Input Vector shape: {final_embeddings.shape}")

print("\n--- BIRINCHI TOKENNING dastlabki 10 TA FLOATING-POINT QIYMATI ---")
print("Floating-point Vektorlar (First 10 values):")
print(final_embeddings[0, 0, :10].detach().numpy())


wte (Token Vector) shape: torch.Size([1, 6, 768])
wpe (Position Vector) shape: torch.Size([1, 6, 768])
Final Input Vector shape: torch.Size([1, 6, 768])

--- BIRINCHI TOKENNING dastlabki 10 TA FLOATING-POINT QIYMATI ---
Floating-point Vektorlar (First 10 values):
[-0.08297435 -0.23652351  0.17846821 -0.01455624  0.11711468 -0.20827545
 -0.18032856 -0.31410712 -0.04306382  0.03620729]


### 4-BOSQICH: PPT DARSIDAGI ASOSIY EKSPERIMENT
**Solishtirish:**
- **1-Gap:** *"Odam kitob o'qidi"*
- **2-Gap:** *"Kitob o'qidi odam"*

**Gipoteza:** Ikkala gapda so'zlar bir xil bo'lgani uchun ularning `wte` (lug'aviy) vektorlari teng, lekin `wpe` (pozitsion) vektorlar qo'shilgach, yakuniy vektorlar orasidagi **Cosine Similarity** o'zgaradi!


In [4]:
#  Odam kitob o'qidi vs Kitob o'qidi odam eksperimenti

text1 = "Odam kitob o'qidi"
text2 = "Kitob o'qidi odam"

ids1 = tokenizer(text1, return_tensors="pt")["input_ids"]
ids2 = tokenizer(text2, return_tensors="pt")["input_ids"]

# Wte va Wpe larini hisoblaymiz
wte1 = model.wte(ids1)
wpe1 = model.wpe(torch.arange(0, ids1.shape[1]).unsqueeze(0))
final1 = wte1 + wpe1

wte2 = model.wte(ids2)
wpe2 = model.wpe(torch.arange(0, ids2.shape[1]).unsqueeze(0))
final2 = wte2 + wpe2

# Cosine Similarity hisoblash funksiyasi
def get_similarity(v1, v2):
    return F.cosine_similarity(v1.flatten(), v2.flatten(), dim=0).item()

sim_wte = get_similarity(wte1, wte2)
sim_final = get_similarity(final1, final2)

print(f"1-Gap Tokenlar: {tokenizer.tokenize(text1)}")
print(f"2-Gap Tokenlar: {tokenizer.tokenize(text2)}")
print("-" * 50)
print(f"Faqat Wte (Lug'at Vektorlari) O'xshashligi: {sim_wte:.4f}")
print(f"Wte + Wpe (Pozitsion Vektor Qo'shilgach) O'xshashlik: {sim_final:.4f}")
print("-" * 50)
print("💡 XULOSA: Pozitsiya kodi (Wpe) qo'shilgani sababli model ikkala gapdagi so'zlar tartibi va grammatik urg'u har xil ekanligini anglab yetdi!")


1-Gap Tokenlar: ['O', 'dam', 'Ġkit', 'ob', 'Ġo', "'", 'q', 'idi']
2-Gap Tokenlar: ['Kit', 'ob', 'Ġo', "'", 'q', 'idi', 'Ġo', 'dam']
--------------------------------------------------
Faqat Wte (Lug'at Vektorlari) O'xshashligi: 0.2120
Wte + Wpe (Pozitsion Vektor Qo'shilgach) O'xshashlik: 0.7474
--------------------------------------------------
💡 XULOSA: Pozitsiya kodi (Wpe) qo'shilgani sababli model ikkala gapdagi so'zlar tartibi va grammatik urg'u har xil ekanligini anglab yetdi!


In [5]:
#  Qwen2.5 Modelining Embedding tuzilishi
qwen_model = AutoModel.from_pretrained("Qwen/Qwen2.5-0.5B")

print("--- QWEN-2.5 MODEL EMBEDDING STRUCTURE ---")
print("Embed Tokens (wte o'rnida):", qwen_model.embed_tokens)
print("\n💡 Qwen2.5 da alohida 'wpe' matritsasi Yo'q! Chunki u RoPE (Rotary Position Embedding) mexanizmidan foydalanadi.")


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

c:\Users\sharg\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sharg\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

--- QWEN-2.5 MODEL EMBEDDING STRUCTURE ---
Embed Tokens (wte o'rnida): Embedding(151936, 896)

💡 Qwen2.5 da alohida 'wpe' matritsasi Yo'q! Chunki u RoPE (Rotary Position Embedding) mexanizmidan foydalanadi.
